In [4]:
import polars as pl 

pl.Config.set_tbl_rows(-1)
pl.Config.set_float_precision(2)
pl.Config.set_decimal_separator(",")
pl.Config.set_thousands_separator(".")

ENDERECO_DADOS = "./../../Dados/"
ENDERECO_VOTACAO ="./../../Dados/"


In [5]:
# Obtendo os dados
try:
    df_bolsa_familia = pl.scan_parquet(ENDERECO_DADOS + "bolsa_familia.parquet")
    # print(df_bolsa_familia)    
    
    df_dados_votacao = pl.read_csv(ENDERECO_VOTACAO + "votacao_secao_2022_BR.csv", separator=";", encoding="iso-8859-1")
    
    # print(df_dados_votacao.columns)
    
    # Filtrar para segundo turno "NR_TURNO" == 2 e nº do candidato "NR_VOTAVEL" 13 e 22
    df_votacao_turno2 = df_dados_votacao.filter(
        (pl.col("NR_TURNO") == 2) & 
        ((pl.col("NR_VOTAVEL").is_in([13, 22])))
    )
    # print(df_votacao_turno2)
    
    
except Exception as e:
    print(f"Erro ao obter os dados: {e}")

#### Processando Votação

In [6]:
# Processando os dados de votação
try:
    # Delimitar as variáveis, converter Categorical, agrupar/totalizar
    
    # Delimitando as variáveis SG_UF, NM_VOTAVEL, QT_VOTOS
    df_votacao = df_votacao_turno2.lazy().select(
        ["SG_UF", "NM_VOTAVEL", "QT_VOTOS"]
    )    
    # print(df_votacao)
    
    # Convertendo para Categorical 
    df_votacao = df_votacao.with_columns(
        pl.col("SG_UF").cast(pl.Categorical),
        pl.col("NM_VOTAVEL").cast(pl.Categorical),
    )
    
    # Agrupar e totalizar os votos por estado e candidato
    df_votacao = (
        df_votacao.group_by(["SG_UF", "NM_VOTAVEL"])
        .agg(pl.sum("QT_VOTOS"))
        .sort("SG_UF", descending=True)
    )
    
    # Executa o plano de execução e coleta os resultado em um DataFrame
    df_votacao = df_votacao.collect()
    print(df_votacao)
    
    
except Exception as e:
    print(f"Erro ao processar os dados: {e}")

shape: (56, 3)
┌───────┬───────────────────────────┬────────────┐
│ SG_UF ┆ NM_VOTAVEL                ┆ QT_VOTOS   │
│ ---   ┆ ---                       ┆ ---        │
│ cat   ┆ cat                       ┆ i64        │
╞═══════╪═══════════════════════════╪════════════╡
│ ZZ    ┆ LUIZ INÁCIO LULA DA SILVA ┆ 152.905    │
│ ZZ    ┆ JAIR MESSIAS BOLSONARO    ┆ 145.264    │
│ TO    ┆ LUIZ INÁCIO LULA DA SILVA ┆ 434.593    │
│ TO    ┆ JAIR MESSIAS BOLSONARO    ┆ 411.654    │
│ SP    ┆ JAIR MESSIAS BOLSONARO    ┆ 14.216.587 │
│ SP    ┆ LUIZ INÁCIO LULA DA SILVA ┆ 11.519.882 │
│ SE    ┆ JAIR MESSIAS BOLSONARO    ┆ 421.086    │
│ SE    ┆ LUIZ INÁCIO LULA DA SILVA ┆ 862.951    │
│ SC    ┆ LUIZ INÁCIO LULA DA SILVA ┆ 1.351.918  │
│ SC    ┆ JAIR MESSIAS BOLSONARO    ┆ 3.047.630  │
│ RS    ┆ JAIR MESSIAS BOLSONARO    ┆ 3.733.185  │
│ RS    ┆ LUIZ INÁCIO LULA DA SILVA ┆ 2.891.851  │
│ RR    ┆ LUIZ INÁCIO LULA DA SILVA ┆ 67.128     │
│ RR    ┆ JAIR MESSIAS BOLSONARO    ┆ 213.518    │
│ RO    ┆ LUIZ I

In [7]:
# Processando os dados de bolsa família
try:
    # Delimitar as variáveis, converter Categorical, agrupar/totalizar
        # Executa o plano de execução e coleta os resultados em um DataFrame
    df_bolsa_familia = df_bolsa_familia.collect()  
    display(df_bolsa_familia)


    # # Delimitando as variáveis 'UF', 'VALOR PARCELA'
    # df_bolsa_familia = df_bolsa_familia.lazy().select(
    #     ['UF', 'VALOR PARCELA']
    # )

    # # Converte para Categorical
    # df_bolsa_familia = df_bolsa_familia.with_columns(
    #     pl.col("UF").cast(pl.Categorical)
    # )

    # # Agrupar e totalizar os valores por estado
    # df_bolsa_familia = (
    #     df_bolsa_familia.group_by('UF')
    #     .agg(pl.sum('VALOR PARCELA'))
    #     .sort('UF', descending=False)
    # )

    # # Executa o plano de execução e coleta os resultados em um DataFrame
    # df_bolsa_familia = df_bolsa_familia.collect()  
    # display(df_bolsa_familia)
    
    
    
except Exception as e:
    print(f"Erro ao processar os dados de bolsa familia: {e}")